Stochasticity
=============

An important aspect of Flatland scenarios will be their **stochasticity**, which means how often and for how long trains will malfunction. Malfunctions force the
agents to reconsider their plans, which can be costly.







## Effects Generators

As described in <a href="../../key-concepts/key_concepts.html">Key Concepts</a> section, effects generators allow to modify the environment state by two hooks:
* `on_episode_step_start`
* `on_episode_step_end`
  
This allows to introduce stochstasticity into a Flatland run.

In contrast to `FlatlandCallbacks` which are used called by a trajectory runner/evaluator, effects generators are called by the simulation directly before and after the env's `step`.

## Malfunctions
Stochastic events are common in railway networks. The initial plan often needs to be rescheduled during operations as minor events such as delayed departure
from train stations, various malfunctions on trains or infrastructure, or even problematic weather lead to delayed trains.

Malfunctions are implemented by `MalfunctionEffectsGenerator` using a [Poisson process](https://en.wikipedia.org/wiki/Poisson_point_process) to simulate delays
by stopping agents at random times for random durations. Trains that malfunction can't move for a random, but known, number of steps. They of course block
the trains following them 😬.

The parameters necessary for the stochastic events are provided as a `NamedTuple` called `MalfunctionParameters`:


In [ ]:
from flatland.envs.malfunction_generators import MalfunctionParameters
stochastic_data = MalfunctionParameters(
    malfunction_rate=1 / 10000,  # Rate of the Poisson process per agent and step
    min_duration=15,  # Malfunctions last at least 15 steps
    max_duration=50  # Malfunctions last at most 50 steps
)
stochastic_data

The parameters are as follows:

- `malfunction_rate` is the rate $\lambda$ of the Poisson process per agent and environment step: in each step, each agent breaks down with probability
  $1 - e^{-\lambda}$ (a draw while it is still malfunctioning is discarded).
- `min_duration` and `max_duration` set the range of malfunction durations: a malfunction lasts a number of steps drawn uniformly from
  `min_duration` to `max_duration` (both included). A drawn 0 (with `min_duration = 0`) is no malfunction.

See *The default malfunction generator `ParamMalfunctionGen`* below for the resulting process and the expected number of malfunctions.

You can then introduce stochasticity in an environment by using the `malfunction_generator` parameter of the `RailEnv` constructor:


```python
from flatland.envs.malfunction_generators import ParamMalfunctionGen
from flatland.envs.rail_env import RailEnv
RailEnv(
    ...
malfunction_generator = ParamMalfunctionGen(stochastic_data),
...
)
```

In your controller, you can then check whether an agent is malfunctioning:

In [ ]:
from flatland.envs.rail_env import RailEnv
from flatland.env_generation.env_generator import env_generator


env, _, _ = env_generator(
    malfunction_interval=1,  # Insanely low interval to show the effect. Inverse of rate of malfunction occurrence. Goes into `ParamMalfunctionGen`.
    malfunction_duration_min=15,  # Minimal duration of malfunction
    malfunction_duration_max=50  # Max duration of malfunction
)
env

action_dict = dict()
obs, rew, done, info = env.step(action_dict)


for a in range(env.get_num_agents()):
    if info['malfunction'][a] > 0:
        # info['malfunction'][a] > 0 while the agent is malfunctioning.
        # Since flatland 5.0.0, the counter is decremented at the start of step(), so the agent is still blocked for info['malfunction'][a] - 1 more steps.
        if info['malfunction'][a] > 1:
            print(f"agent {a} is malfunctioning and can't move for {info['malfunction'][a] - 1} more steps!")
        else:
            print(f"agent {a} is malfunctioning in this step and can move again in the next step!")

You will quickly realize that this will lead to unforeseen difficulties which means that your controller needs to observe the environment at all times to be
able to react to the stochastic events! In particular, `info['action_required']` is `False` during a malfunction and when it ends: give a moving action
to get the train moving again (see [Actions](actions)).

<!-- [Check out the starter kit](https://gitlab.aicrowd.com/flatland/neurips2020-flatland-starter-kit/blob/master/reinforcement_learning/multi_agent_training.py#L55) for a complete example of how to train a model using malfunctions. -->

### The default malfunction generator `ParamMalfunctionGen`

`ParamMalfunctionGen` from `flatland.envs.malfunction_generators` is the malfunction generator of the Flatland challenges and of `env_generator`
(its `malfunction_interval` is $1/\lambda$). `RailEnv` wraps the generator passed as `malfunction_generator` into a `MalfunctionEffectsGenerator`. Without a
`malfunction_generator`, `RailEnv` uses `NoMalfunctionGen`, a `ParamMalfunctionGen` with rate 0, which never generates a malfunction.

#### What happens in a step

At the start of every `step()`, `RailEnv` goes through all agents in handle order and draws once from the generator for each: with probability
$p = 1 - e^{-\lambda}$, the draw is a malfunction with a duration of $D$ steps, drawn uniformly from `min_duration` to `max_duration` (both
included). A drawn $D = 0$ is no malfunction.

The draw only takes effect if the agent's malfunction counter is 0, that is, if the agent is not malfunctioning. A draw while the agent is still
malfunctioning is discarded: malfunctions neither stack nor get longer. The draw is made all the same and uses up its random numbers, so the draws of a
step don't depend on what happened before. With the same seed, the malfunctions therefore don't depend on the agents' actions, as long as nothing else
draws from the env's random state during the steps. If the draw takes effect, the agent can't move for exactly $D$ steps, starting with the current one:
it is in `MALFUNCTION` (or `MALFUNCTION_OFF_MAP` if it hasn't departed yet). An agent can break down again in the first step after a malfunction ends.

A moving agent loses $D + 1$ steps, as long as it gets back to its max speed in one step: see *Malfunction duration and steps lost* below. Up to
flatland-rl 4.3.0, the generator added 1 to the drawn duration
([flatland-rl#565](https://github.com/flatland-association/flatland-rl/issues/565)): a malfunction blocked the agent for $D + 1$ steps.

```{admonition} Pro memoria
:class: note

A malfunction of duration $D \geq 1$, drawn uniformly from `min_duration` to `max_duration` (both included) at the start of the step to
`elapsed_steps = t`:

- **blocked:** the agent is in `MALFUNCTION` (or `MALFUNCTION_OFF_MAP`) after the steps to $t, \ldots, t + D - 1$, exactly $D$ steps. The action given
  for the step to $t$ already has no effect.
- **counter:** `info['malfunction']` is $D, \ldots, 1$ after these steps, and 0 from the step to $t + D$ on.
- **after the malfunction:** in the step to $t + D$, a moving action gives the agent its speed back, but, as after any stop, it only moves from the step
  to $t + D + 1$ on.
- **steps lost:** a moving agent that gets back to its max speed in one step (as with the default `acceleration_delta = 1`) arrives $D + 1$ steps later
  than without the malfunction. With a slower acceleration, it can lose more. An agent that hasn't departed yet loses at most $D$ steps.
- A drawn $D = 0$ is no malfunction.
```

```{note}
**flatland-rl 4.3.0 and 5.0.0** draw malfunctions in the same steps, with the same $D$: the generator is the same code, except that 4.3.0 adds 1 to
$D$, so a malfunction blocks the agent for $D + 1$ steps instead of $D$. Where `step()` decrements the counter differs, an implementation detail that
shows in `info['malfunction']`, and in 5.0.0's speed model an agent only moves from the step after its speed is back. For the same parameters, a moving
agent loses as many steps in both:

|                                                 | 5.0.0                                                 | 4.3.0                                                              |
|-------------------------------------------------|-------------------------------------------------------|--------------------------------------------------------------------|
| steps the agent is blocked                      | $D$                                                   | $D + 1$                                                            |
| counter decremented                             | at the start of `step()`, before the draw             | at the end of `step()`, after the agent's state update             |
| `info['malfunction']` over a malfunction        | counts down from $D$ to 1                             | counts down from $D$ to 0                                          |
| `info['malfunction']` after a step              | number of steps the agent is blocked, including this one | number of steps the agent is still blocked after this one          |
| last step of a malfunction                      | `info['malfunction']` is 1                            | `info['malfunction']` is already 0 while the agent is malfunctioning |
| first step after a malfunction                  | speed is back, but the agent only moves from the next step on, as after any stop | the agent moves again                      |
| steps a moving agent loses                      | $D + 1$                                               | $D + 1$                                                            |
| malfunction that ends before the agent departs  | no effect on the departure                            | the agent skips `READY_TO_DEPART` and enters the map one step earlier than without it |
```

Every agent draws in every step, whatever it is doing:

* The probability doesn't depend on the agent's behaviour: moving, stopped or waiting agents break down equally often.
* Agents that haven't departed yet break down as well (`MALFUNCTION_OFF_MAP`) and can't depart while malfunctioning.
* Agents that are `DONE` keep drawing too. A malfunction doesn't change their state, but it still sets `info['malfunction']` and increases
  `agent.malfunction_handler.num_malfunctions`.


The plots show a malfunction with $D = 2$ of an agent with max speed 1/4 and `earliest_departure = 4`, given `MOVE_FORWARD` in every step, with
flatland-rl 5.0.0 and 4.3.0 (`env_generator(n_agents=1, seed=14, malfunction_interval=6, malfunction_duration_min=2, malfunction_duration_max=2)`,
which makes the same draws in both). The 5.0.0 plot is computed from a run when the book is built, the 4.3.0 plot is recorded. They use the same conventions as the plots in [Speed Profiles](speed_profiles.ipynb): speed (left axis) and distance
travelled (right axis) after each step, the arrows mark a step by the `elapsed_steps` it starts from, with a comment, and the bars below the x-axis show
the state after each step: in 5.0.0 as `agent.state` with `RailEnvStateMachineWrapper` (wrapped, the same as `info['state']`) and without (unwrapped),
in 4.3.0 as `agent.state`. Below the states:

* malfunction draw: the draw at the start of the step from that `elapsed_steps` on, one per step. Grey dots draw no malfunction, the filled dot draws a
  malfunction that takes effect (dotted line), the hollow dot one that is discarded because the agent is still malfunctioning.
* `info['malfunction']` after each step.

In both versions, the malfunction drawn at the start of the step from 8 to 9 stops the agent already in that step: it doesn't move, and its speed is 0.
The versions differ in four places:

* the steps blocked: in 5.0.0, the agent is in `MALFUNCTION` (shaded) after the steps to 9 and 10, that is, for $D = 2$ steps; in 4.3.0, after the
  steps to 9, 10 and 11, for $D + 1 = 3$ steps.
* the counter: 2, 1 in 5.0.0, and 2, 1, 0 in 4.3.0 (see the note above).
* after the malfunction: in 5.0.0, `MOVE_FORWARD` gives the agent speed 1/4 again in the step from 10 to 11, but, as for a stopped train, it only moves
  from the next step on. In 4.3.0, it moves again in the step from 11 to 12. So the malfunction costs the agent $D + 1 = 3$ steps in both.
* the departure: in 4.3.0, the agent is `READY_TO_DEPART` after the step to 4 and enters the map one step later than in 5.0.0. So after 18 steps, it is at
  distance 5/2 in 4.3.0, and a quarter of a cell further, at 11/4, in 5.0.0.

In [ ]:
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb

from flatland.env_generation.env_generator import env_generator
from flatland.envs.malfunction_effects_generators import MalfunctionEffectsGenerator
from flatland.envs.rail_env_action import RailEnvActions

SPEED_COLOR, DISTANCE_COLOR, INK, MUTED, GRID = "#1d4ed8", "#b45309", "#1f2937", "#6b7280", "#e5e7eb"
DEPARTURE_COLOR, MALFUNCTION_COLOR = "#dc2626", "#7c3aed"
# state: (fill, faded out to the right, as a state holds from its first elapsed_steps up to just before the next state;
# start line, at the first elapsed_steps of the state)
STATE_COLORS = {"WAITING": ("#e5e7eb", "#6b7280"), "READY_TO_DEPART": ("#fde68a", "#d97706"),
                "MOVING": ("#bbf7d0", "#16a34a"), "MALFUNCTION": ("#ddd6fe", "#7c3aed")}
T, X_MAX = 4, 18  # t = earliest_departure, last elapsed_steps shown
SPEED_MAX, DISTANCE_MAX = 0.4, 3.2  # same ratio, so that speed 1/4 and distance 2 are at the same height


def plot_malfunction_trajectory(speed, travelled, states, draws, malfunction, labels, title):
    """Speed (left axis) and distance travelled (right axis) per elapsed_steps 0 to X_MAX, as steps: a value set by the
    step to elapsed_steps = k holds from k up to just before k+1; a fine line interpolates it during the step, from k-1
    to k. Off the map, speed and distance are None, shown as 0.
    states: per state view, (from elapsed_steps, to elapsed_steps, state), drawn as rows of bars below the x axis.
    draws: per elapsed_steps k, the outcome of the malfunction draw at the start of the step k -> k+1: None (no
    malfunction), "effect" (malfunction drawn, takes effect) or "discarded" (malfunction drawn while malfunctioning).
    malfunction: info['malfunction'] after the step to elapsed_steps = k, for k = 0 to X_MAX.
    labels: (k, comment on the step k -> k+1, y of the arrow tip, y of the text on the speed axis, axis of the arrow tip)."""
    xs = list(range(X_MAX + 1))
    fig, (ax_s, ax_states) = plt.subplots(2, 1, figsize=(11, 7.4), sharex=True,
                                          gridspec_kw=dict(height_ratios=[6, 2.4], hspace=0.08))
    ax_d = ax_s.twinx()
    speed, travelled = [float(v) for v in speed], [float(v) for v in travelled]
    line_s, = ax_s.step(xs, speed, where="post", color=SPEED_COLOR, linewidth=2, label="speed (cells per step, left)")
    line_d, = ax_d.step(xs, travelled, where="post", color=DISTANCE_COLOR, linewidth=2,
                        label="distance travelled (cells, right)")
    ax_s.plot(xs, speed, color=SPEED_COLOR, linewidth=0.8)
    ax_d.plot(xs, travelled, color=DISTANCE_COLOR, linewidth=0.8)
    line_t = ax_s.axvline(T, color=DEPARTURE_COLOR, linestyle="--", linewidth=1.5, label=f"earliest_departure = {T}")
    # malfunction: drawn at the start of a step (dotted), in MALFUNCTION from the first to the last step it blocks (shaded)
    effect = [k for k, d in enumerate(draws) if d == "effect"]
    for k in effect:
        ax_s.axvline(k, color=MALFUNCTION_COLOR, linestyle=":", linewidth=1.5)
    start, end = None, None
    for view, spans in states.items():
        for s, e, state in spans:
            if state == "MALFUNCTION":
                start, end = s, e
    shade = ax_s.axvspan(start, end, color=STATE_COLORS["MALFUNCTION"][0], alpha=0.35, linewidth=0,
                         label=f"MALFUNCTION (elapsed_steps {start} to {end - 1})")
    line_m = ax_s.plot([], [], color=MALFUNCTION_COLOR, linestyle=":", linewidth=1.5,
                       label="malfunction drawn (start of the step)")[0]
    ax_s.set_xlim(0, X_MAX)
    ax_s.set_ylim(0, SPEED_MAX)
    ax_d.set_ylim(0, DISTANCE_MAX)
    ax_s.set_yticks([0, 0.125, 0.25, 0.375], ["0", "1/8", "1/4", "3/8"])
    ax_d.set_yticks([0, 0.5, 1, 1.5, 2, 2.5, 3])
    ax_s.set_ylabel("speed (cells per step)", color=INK)
    ax_d.set_ylabel("distance travelled (cells)", color=INK)
    for ax in (ax_s, ax_d):
        ax.spines["top"].set_visible(False)
        ax.tick_params(colors=INK, labelsize=9)
    ax_s.spines["left"].set_color(SPEED_COLOR)
    ax_s.spines["right"].set_visible(False)
    ax_d.spines["left"].set_visible(False)
    ax_d.spines["right"].set_color(DISTANCE_COLOR)
    ax_s.grid(axis="x", color=GRID, linewidth=0.8)
    ax_s.set_axisbelow(True)
    for k, text, y_tip, y_text, axis in labels:
        tip = (k, y_tip) if axis == "speed" else ax_s.transData.inverted().transform(ax_d.transData.transform((k, y_tip)))
        ax_s.annotate(text, xy=tuple(tip), xytext=(k + 0.06, y_text), fontsize=8.5, color=INK, ha="left", va="bottom",
                      arrowprops=dict(arrowstyle="->", color=MUTED, linewidth=1, shrinkA=0, shrinkB=2, relpos=(0, 0)),
                      bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor=GRID, linewidth=0.8))
    fig.legend(handles=[line_s, line_d, line_t, line_m, shade], loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=3,
               frameon=False, fontsize=9, labelcolor=INK)
    ax_s.set_title(title, color=INK, fontsize=11, loc="left")

    # rows below the x axis, top down: state after each step per state view, malfunction draws, info['malfunction']
    rows = list(states) + ["malfunction draw", "info['malfunction']"]
    n = len(rows)
    for row, (view, spans) in enumerate(states.items()):
        y0, y1 = n - row - 1, n - row
        for s, e, state in spans:
            fill, edge = STATE_COLORS[state]
            gradient = np.dstack([np.broadcast_to(to_rgb(fill), (1, 64, 3)), np.linspace(1, 0.1, 64).reshape(1, 64, 1)])
            ax_states.imshow(gradient, extent=(s, e, y0 + 0.06, y1 - 0.06), aspect="auto", interpolation="bilinear",
                             zorder=1)
            ax_states.plot([s, s], [y0 + 0.06, y1 - 0.06], color=edge, linewidth=2.5, zorder=2)
            short = e - s < 2
            ax_states.text((s + e) / 2, (y0 + y1) / 2, state.replace("_TO_", "_TO_\n") if short else state,
                           ha="center", va="center", fontsize=6.5 if short else 8, color=INK, linespacing=1.0, zorder=3)
    y_draw, y_counter = 1.5, 0.5
    for k, d in enumerate(draws):
        if d is None:
            ax_states.plot(k, y_draw, "o", color=MUTED, markersize=3, zorder=3)
        else:
            ax_states.plot(k, y_draw, "o", markersize=7, zorder=3, markeredgewidth=1.5, markeredgecolor=MALFUNCTION_COLOR,
                           color=MALFUNCTION_COLOR if d == "effect" else "white")
            ax_states.text(k - 0.2 if d == "effect" else k + 0.2, y_draw, "takes effect" if d == "effect" else "discarded",
                           ha="right" if d == "effect" else "left", va="center", fontsize=7, color=MALFUNCTION_COLOR, zorder=4,
                           bbox=dict(boxstyle="square,pad=0.1", facecolor="white", edgecolor="none"))
    for k, m in enumerate(malfunction):
        ax_states.text(k, y_counter, str(m), ha="center", va="center", fontsize=8,
                       color=MALFUNCTION_COLOR if m > 0 else MUTED, zorder=3)
    for k in effect:
        ax_states.axvline(k, color=MALFUNCTION_COLOR, linestyle=":", linewidth=1.5, zorder=4)
    ax_states.axvline(T, color=DEPARTURE_COLOR, linestyle="--", linewidth=1.5, zorder=4)
    ax_states.set_ylim(0, n)
    ax_states.set_yticks([n - row - 0.5 for row in range(n)], rows, fontsize=8)
    ax_states.tick_params(axis="y", length=0, pad=10)
    for side in ("top", "left", "right"):
        ax_states.spines[side].set_visible(False)
    ax_states.set_xticks(xs)
    ax_states.tick_params(colors=INK, labelsize=9)
    ax_states.set_xlabel("elapsed_steps", color=INK)
    plt.show()


def run_malfunction(malfunction_interval=6):
    """The agent of the plots in flatland-rl as installed, given MOVE_FORWARD in every step up to X_MAX. Returns per elapsed_steps:
    speed, distance travelled, state with and without RailEnvStateMachineWrapper, the outcome of the malfunction draw at the start
    of the step k -> k+1 (None, "effect" or "discarded") and info['malfunction']."""
    env, _, _ = env_generator(n_agents=1, seed=14, malfunction_interval=malfunction_interval, malfunction_duration_min=2,
                              malfunction_duration_max=2, skip_state_machine_update=False)
    agent = env.agents[0]
    generator = next(g for g in env.effects_generator.effects_generators if isinstance(g, MalfunctionEffectsGenerator)).malfunction_generator
    generate, positive = generator.generate, []

    def recording_generate(np_random):
        malfunction = generate(np_random)
        positive.append(malfunction is not None and malfunction.num_broken_steps > 0)
        return malfunction

    generator.generate = recording_generate
    speed, travelled, wrapped, unwrapped, draws, malfunction = [0], [0], [agent.state.name], [agent.derived_state().name], [], [0]
    cells, cell = 0, None
    for _ in range(X_MAX):
        blocked_before = agent.malfunction_handler.malfunction_down_counter
        _, _, _, info = env.step({0: RailEnvActions.MOVE_FORWARD})
        # a positive draw takes effect unless the agent is still malfunctioning after the counter is decremented
        draws.append(None if not positive[-1] else "discarded" if blocked_before > 1 else "effect")
        if agent.current_entry_point is not None:
            cells += cell is not None and agent.current_entry_point[0] != cell
            cell = agent.current_entry_point[0]
        speed.append(agent.speed_counter.speed or 0)
        travelled.append(0 if cell is None else cells + agent.speed_counter.distance)
        wrapped.append(agent.state.name)
        unwrapped.append(agent.derived_state().name)
        malfunction.append(info["malfunction"][0])
    return speed, travelled, wrapped, unwrapped, draws + [None], malfunction


def spans(states):
    """(from elapsed_steps, to elapsed_steps, state) for each run of equal states."""
    starts = [k for k in range(len(states)) if k == 0 or states[k] != states[k - 1]]
    return [(s, e, states[s]) for s, e in zip(starts, starts[1:] + [X_MAX])]


q = Fraction(1, 4)
speed, travelled, wrapped, unwrapped, draws, malfunction = run_malfunction()
# D = 2: blocked in the steps to 9 and 10, counter 2, 1; a moving agent loses D + 1 = 3 steps against a run without malfunctions
assert draws[8:10] == ["effect", "discarded"] and malfunction[8:12] == [0, 2, 1, 0], (draws, malfunction)
assert [k for k, state in enumerate(wrapped) if state == "MALFUNCTION"] == [9, 10], wrapped
assert (run_malfunction(malfunction_interval=None)[1][X_MAX] - travelled[X_MAX]) / q == 2 + 1
plot_malfunction_trajectory(
    speed=speed, travelled=travelled,
    states={"wrapped": spans(wrapped), "unwrapped": spans(unwrapped)},
    draws=draws, malfunction=malfunction,
    labels=[(8, "malfunction drawn, D = 2: blocked for D = 2 steps,\nno move in this step, speed 0, distance unchanged", 1, 0.37, "distance"),
            (9, "malfunction drawn again: discarded,\nas the agent is still malfunctioning", 0, 0.31, "speed"),
            (10, "malfunction over: speed 1/4,\ndistance unchanged", 0, 0.275, "speed"),
            (11, "moves again by 1/4", 1.25, 0.2, "distance")],
    title="malfunction with D = 2, max speed 1/4, MOVE_FORWARD in every step (flatland-rl 5.0.0)")
plot_malfunction_trajectory(
    speed=[0] * 5 + [q] * 4 + [0] * 3 + [q] * 7,
    travelled=[0] * 6 + [q, 2 * q, 3 * q] + [3 * q] * 3 + [1, 1 + q, 1 + 2 * q, 1 + 3 * q, 2, 2 + q, 2 + 2 * q],
    states={"state": [(0, 4, "WAITING"), (4, 5, "READY_TO_DEPART"), (5, 9, "MOVING"), (9, 12, "MALFUNCTION"), (12, 18, "MOVING")]},
    draws=[None] * 8 + ["effect", "discarded"] + [None] * 8,
    malfunction=[0] * 9 + [2, 1, 0] + [0] * 7,
    labels=[(8, "malfunction drawn, D = 2: blocked for D + 1 = 3 steps,\nno move in this step, speed 0, distance unchanged", 0.75, 0.37, "distance"),
            (9, "malfunction drawn again: discarded,\nas the agent is still malfunctioning", 0, 0.31, "speed"),
            (11, "malfunction over: moves again by 1/4\nin this step, info['malfunction'] already 0", 0, 0.255, "speed")],
    title="malfunction with D = 2, max speed 1/4, MOVE_FORWARD in every step (flatland-rl 4.3.0)")

#### Edge cases: no malfunctions and a malfunction in every step

* **No malfunctions:** with `malfunction_rate = 0` (or a negative rate), $p$ is exactly 0 and no agent ever breaks down. This is `NoMalfunctionGen`,
  `MalfunctionParameters(0, 0, 0)`. In `env_generator`, `malfunction_interval=0` (or `None`) gives no malfunctions as well: it is replaced by `sys.maxsize`,
  and $1 - e^{-1/\mathtt{sys.maxsize}}$ is exactly 0 in floating point. The agents still draw in every step, but no draw is a malfunction.
* **A malfunction in every step:** with `malfunction_rate = float("inf")`, $p = 1$ and every draw is a malfunction. Any rate from about 37.5 on gives the
  same, as $1 - e^{-\lambda}$ is then exactly 1.0 in floating point. In `env_generator`, this needs a `malfunction_interval` below 1/37.5, for example
  `0.01` (it is typed `int`, but a float works); an interval of at least 1 gives a rate of at most 1, that is, $p \le 1 - e^{-1} \approx 0.63$. As an agent
  breaks down again in the first step after a malfunction ends, it is malfunctioning in every step and never moves: as all agents start off the map, no agent
  ever enters the grid, and all stay in `MALFUNCTION_OFF_MAP` for the whole episode (flatland-rl 4.3.0 and 5.0.0). The draws during a malfunction are still discarded, so an agent has a new malfunction every $D$ steps: with
  `min_duration = max_duration = 2` ($D = 2$), `info['malfunction']` cycles 2, 1, and the agent has 15 malfunctions in 30 steps (in flatland-rl 4.3.0,
  which blocks $D + 1 = 3$ steps: 2, 1, 0, and 10 malfunctions). The expected values below cover this case too: the free periods have length $\frac{1}{p} - 1 = 0$, so $\mu = E[D]$ and the agent is
  malfunctioning 100% of the time.

#### Malfunction duration and steps lost

```{admonition} Semantics: is the drawn duration the number of steps lost?
:class: warning

Take the semantics that the drawn malfunction duration is the number of steps an agent loses. Let $D$ be the value the generator draws uniformly from
`min_duration` to `max_duration` (both included). Three effects make the steps a moving agent actually loses differ from $D$:

| effect                                                                                                                                                                                                                                                  | steps lost | 5.0.0 | 4.3.0 | bug?                                                                                                                                                                                                                       |
|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------|-------|-------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **+1 on the drawn duration:** the generator adds 1 to $D$, and the agent is blocked for $D + 1$ steps.                                                                                                                                                   | +1         | no    | yes   | bug, fixed in 5.0.0 ([flatland-rl#565](https://github.com/flatland-association/flatland-rl/issues/565)): the docstrings say "duration uniformly in [min_duration, max_duration]". |
| **Immediate effect:** the malfunction blocks the agent already in the step at whose start it is drawn, so the action given for that step has no effect.                                                                                                  | 0          | yes   | yes   | bug? It changes which steps are lost, not how many: starting in the next step instead, the agent would still lose as many steps. The +1 may have been added for this step: in 4.3.0, `info['malfunction']` after it is $D$, the number of steps the agent is still blocked after this one. |
| **Acceleration after the malfunction:** in the first step after the malfunction, the agent's speed is back, but it only moves from the next step on.                                                                                                      | +1         | yes   | no    | no, by design: the same as after any stop.                                                                                                                                                                                |
| **steps a moving agent loses in total**                                                                                                                                                                                                                  |            | $D + 1$ | $D + 1$ |                                                                                                                                                                                                       |

The plots above show this for $D = 2$: the agent loses 3 steps in both versions. In 5.0.0, the same holds for agents of any max speed that get back to
their max speed in one step, as with the default `acceleration_delta = 1`. With a slower acceleration, an agent can lose more: with max speed 1 and
`acceleration_delta = 1/4`, $D + 2$ steps. For an agent that
is not moving, for example one that hasn't departed yet, the steps lost depend on what it would have done otherwise: a malfunction that ends before the agent
departs costs no steps in 5.0.0 (in 4.3.0 the agent even departs one step earlier, see the note above).
```

#### The resulting process

$p = 1 - e^{-\lambda}$ is the probability that a [Poisson process](https://en.wikipedia.org/wiki/Poisson_point_process) with rate $\lambda$ has at least one
event in an interval of length 1. So the malfunction draws of an agent are a Poisson process observed at integer steps (a Bernoulli process): several events
within one step count as one, and $p < \lambda$ (for small $\lambda$, $p \approx \lambda$). The draws of different agents are independent.

As each draw is independent of the earlier ones, discarding the draws during a malfunction is the same as not drawing at all: $\lambda$ is the failure
rate of a train while it is not malfunctioning (the clock is paused during a repair), not the rate of malfunctions overall. Each agent's malfunctions
alternate between free and broken periods (an alternating renewal process):

* free: the number of steps from the end of one malfunction to the start of the next is geometric with mean $\frac{1}{p} - 1 \approx \frac{1}{\lambda} - \frac{1}{2}$,
* broken: $D$ steps with mean $E[D] =$ `(min_duration + max_duration) / 2` (for `min_duration` of at least 1: a drawn 0 is no malfunction).

A malfunction starts on average every $\mu = \frac{1}{p} - 1 + E[D]$ steps. The expected values below are exact for the number of positive draws
($T \cdot p$ in $T$ steps) and as long-run averages for the malfunctions; for a finite number of steps $T$, $\frac{T}{\mu}$ is an approximation, as
every agent starts free.

#### Expected values

Per agent:

| quantity                                                         | expected value                                                  |
|------------------------------------------------------------------|-----------------------------------------------------------------|
| probability of a malfunction draw in a step                      | $p = 1 - e^{-\lambda}$                                          |
| steps between the end of a malfunction and the start of the next | $\frac{1}{p} - 1 \approx \frac{1}{\lambda} - \frac{1}{2}$       |
| steps between the starts of two consecutive malfunctions         | $\mu = \frac{1}{p} - 1 + E[D]$                                  |
| number of malfunctions in $T$ steps                              | $\approx \frac{T}{\mu}$ (long-run average, for $T \gg \mu$)     |
| fraction of steps in malfunction                                 | $\frac{E[D]}{\mu}$ (long-run average)                           |

For $n$ agents, multiply the number of malfunctions by $n$.

So the expected number of malfunctions in $T$ steps is less than $\lambda T$: $1/\lambda$ is not the mean number of steps between malfunctions, but
roughly the mean number of steps an agent runs free between two malfunctions. The difference matters when the durations are not small compared to
$1/\lambda$. For the `env_generator` defaults (`malfunction_interval=540`, `malfunction_duration_min=20`, `malfunction_duration_max=50`):
$p = 0.00185$, $1/p - 1 = 539.5$, $E[D] = 35$, $\mu = 574.5$, so an agent has on average $1000/574.5 \approx 1.74$ malfunctions per 1000 steps (not
$1000/540 \approx 1.85$) and is malfunctioning $35/574.5 \approx 6.1\%$ of the time.

The expected values for `stochastic_data` from above:

In [ ]:
import math

p = 1 - math.exp(-stochastic_data.malfunction_rate)
mean_duration = (stochastic_data.min_duration + stochastic_data.max_duration) / 2
mean_free = 1 / p - 1
mu = mean_free + mean_duration
print(f"probability of a malfunction draw per agent and step: {p:.6f}")
print(f"mean malfunction duration: {mean_duration} steps")
print(f"mean steps between the end of a malfunction and the start of the next: {mean_free:.1f}")
print(f"mean steps between the starts of two consecutive malfunctions: {mu:.1f}")
print(f"expected malfunctions per agent in 1000 steps: {1000 / mu:.3f}")
print(f"fraction of steps in malfunction: {mean_duration / mu:.2%}")

#### What is not guaranteed

* These are expected values, not bounds: there is no minimum or maximum number of malfunctions per agent or episode. An agent may never break down, or
  break down several times in a row.
* The malfunction draws come from a block of 1,000,000 random numbers that the generator takes from the env's random state at its first draw. They are
  used in order, one per agent and step, and the block is reused cyclically: after 1,000,000 draws (for example 100 agents × 10,000 steps), the draws
  repeat.
* The block is not drawn anew on `env.reset()`: an env created with the same seed reproduces the malfunctions, but resetting an env with the same seed
  continues with the next draws of the block and gives different malfunctions. The durations, in contrast, are drawn from the env's random state at
  each malfunction draw. This is a bug, to be fixed in flatland-rl 5.0.0
  ([flatland-rl#564](https://github.com/flatland-association/flatland-rl/issues/564)): then a reset with the same seed reproduces the malfunctions.

## Conditional Malfunctions

`ConditionalMalfunctionEffectsGenerator` Generate agent malfunctions conditionally with conditional rate and duration:
* `earliest_malfunction`
* `max_num_malfunctions`
* `condition` : `MalfunctionCondition` `EnvAgent`

A `MalfunctionCondition` takes an `EnvAgent` and the elapsed steps and decides whether a malfunction should be drawn from the Poisson process.
Since `flatland 5.0.0`, it is a `Callable[[EnvAgent, int, bool], bool]`: the third argument `previous_in_malfunction` tells whether the agent was in
malfunction at the end of the previous step. Conditions are evaluated at the start of the step, after the malfunction counter has been decremented, so a
condition on the agent's state should use `agent.derived_state(in_malfunction=previous_in_malfunction)`, as the pre-defined ones do. Custom conditions
with two arguments written for `flatland 4.x` need the third argument.

There are 3 pre-defined such conditions, which can be combined with `make_multi_malfunction_condition` (any of them applies):

* `on_map_state_condition`: is agent on map?
* `condition_stopped_intermediate_and_range`: is agent stopped at an intermediate waypoint and in range of timesteps?
* `condition_stopped_cells_and_range`: is agent stopped on any given cell and during range of timesteps`

This feature allows to generate minor disruptions with faster recovery times at train stations.

> This feature was introduced in [4.2.1](https://github.com/flatland-association/flatland-rl/pull/263)


## Observation Perturbations

Another stochastic feature are "noisy observations". They do not change the behaviour of the simulation/environment but introduce noise in the observation passed to the agent or controller, see [Figure 3.1: The agent–environment interaction in reinforcement learning Sutton and Burton: Reinforcement Learning, 2nd edition](http://incompleteideas.net/book/RLbook2020.pdf).

Find more in the section on observation perturbations in the
<a href="../observation_builder.html">Observations</a> section.